In [ ]:
# v15: smoke, check, generate, train.
#
# Pass criteria before committing to the full run:
#   role adjacency gap  < 0.05   (v14 +0.235, corpus -0.009)
#   pairs per sentence  > 5.0    (v14 3.37, corpus 6.29)
#   hard negative rate  0.4-0.6  (must not have moved)
#   count-rule F1       < 0.35
# If the role gap has not moved, the scene-level framing did not work and you know in
# an hour rather than after a full cycle.

import os, json, importlib
from collections import Counter
from openai import OpenAI

import ddi.prompt, ddi.resolve, ddi.gates, ddi.synth, ddi.divergence
for m in (ddi.synth, ddi.prompt, ddi.resolve, ddi.gates, ddi.divergence):
    importlib.reload(m)

from ddi.data import build_human
from ddi.vocab import build_vocab
from ddi.prompt import make_v14_specs, render_v14, make_v14_sample_fn, v14_fingerprint
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi import gates, divergence as dv

MODEL, API, EFFORT = "gpt-oss-120b", "responses", "low"
V14_ID = "20260807-123340-ff79db"

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=60.0)
client.responses.create(model=MODEL, input="ping", max_output_tokens=16,
                        reasoning={"effort": "low"})
print(f"{MODEL} up. prompt sha {v14_fingerprint()}")

vocab = build_vocab()
train, dev, val = build_human()
per_sent = Counter(r["sent_id"] for r in train)
monsters = {s for s, n in per_sent.items() if n >= 190}
train_f = [r for r in train if r["sent_id"] not in monsters]
v14, _ = load_dataset(V14_ID)


In [ ]:
import ddi.prompt, ddi.resolve, ddi.gates, ddi.synth, ddi.divergence
for m in (ddi.synth, ddi.prompt, ddi.resolve, ddi.gates, ddi.divergence):
    importlib.reload(m)


# ===========================================================================
# Cell 1: render 30 and read them. Free. Do not skip.
# Looking for: does a k=7 or k=8 spec look writable, does the scene carry the
# non-participants without a per-drug role, does MECHANISM content read coherently
# with mode + enzyme + figure together.
# ===========================================================================
specs = make_v14_specs(30, vocab=vocab, seed=0, composition="prior")
print(Counter(len(s["entities"]) for s in specs))
for s in specs[:8]:
    print(render_v14(s), "\n" + "-" * 60)

mech = [s for s in specs if s["asserts"] and s["asserts"][0]["label"] == "MECHANISM"]
for s in mech[:4]:
    print(render_v14(s), "\n" + "-" * 60)



In [ ]:
import importlib, ddi.prompt
importlib.reload(ddi.prompt)
from ddi.prompt import make_v15_specs, render_v15, v15_fingerprint
print(v15_fingerprint())
print([n for n in dir(ddi.prompt) if "unusable" in n])

In [ ]:
from collections import Counter
specs = make_v15_specs(500, vocab=vocab, seed=0)
c = Counter(len(s["entities"]) for s in specs)
print(sorted(c.items()), sum(k*v for k,v in c.items())/500)

In [ ]:
import random, importlib, ddi.prompt
importlib.reload(ddi.prompt)
from ddi.prompt import _unusable

rng = random.Random(0)
sample = [vocab.sample(1, rng)[0] for _ in range(2000)]
bad = [s for s in sample if _unusable(s)]
print(f"{len(bad)/2000:.3f} unusable")
print(bad[:20])

In [ ]:
import random
from ddi.prompt import _unusable
rng = random.Random(0)
fails = 0
for _ in range(300):
    surf, seen = [], set()
    for _ in range(200):
        if len(surf) == 8: break
        c = vocab.sample(1, rng)[0]
        if _unusable(c): continue
        low = c.lower()
        if low in seen or any(low in s or s in low for s in seen): continue
        seen.add(low); surf.append(c)
    if len(surf) < 8: fails += 1
print(f"{fails}/300 failed to reach k=8")

In [ ]:

# ===========================================================================
# Cell 2: smoke 300
# ===========================================================================
GEN = "v15-check"
specs = make_v14_specs(300, vocab=vocab, seed=0, composition="prior")
generate_raw(specs, make_v14_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                       api=API),
             gen_id=GEN, max_workers=16)

did, stats = build_dataset_from_raw(
    GEN, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": v14_fingerprint(), "model": MODEL,
               "reasoning_effort": EFFORT, "composition": "prior", "version": "v15",
               "note": "scene-level roles, entity cap 8, corpus-weighted MECHANISM"},
    vocab_source=vocab.fingerprint(), seed=0)
print(stats["reject_reasons"])
inst, _ = load_dataset(did)

gap = dv.compare(train_f, inst, match_size=True)
print(gap.to_string(index=False))

v14_gap = dv.compare(train_f, v14, match_size=True)
comp = gap.merge(v14_gap[["measurement", "synth"]], on="measurement",
                 suffixes=("_v15", "_v14"))
print("\n" + comp[["measurement", "corpus", "synth_v14", "synth_v15"]]
      .to_string(index=False))

print("""
PASS if:  role adjacency gap < 0.05, pairs per sentence > 5.0,
          hard negative rate 0.4-0.6, count-rule F1 < 0.35
""")


In [ ]:


# ===========================================================================
# Cell 3: read the output, especially high-k and MECHANISM
# ===========================================================================
shown_k, shown_m = 0, 0
for line in (RAW / f"{GEN}.jsonl").read_text().splitlines():
    r = json.loads(line)
    if r.get("error"):
        continue
    s, txt = r["spec"], r["sample"]["sentence"]
    k = len(s["entities"])
    lab = s["asserts"][0]["label"] if s["asserts"] else "NONE"
    if k >= 6 and shown_k < 6:
        print(f"[k={k} {lab}] {txt}\n")
        shown_k += 1
    elif lab == "MECHANISM" and shown_m < 6:
        print(f"[MECH] {txt}")
        print(f"   {s['asserts'][0]['content']}\n")
        shown_m += 1



In [ ]:

# ===========================================================================
# Cell 4: full run, 6000 specs, ~30 min. Only if cell 2 passed.
# ===========================================================================
GEN_FULL = "v15-full"
specs = make_v14_specs(6000, vocab=vocab, seed=0, composition="prior")
generate_raw(specs, make_v14_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                       api=API),
             gen_id=GEN_FULL, max_workers=16)

v15_id, stats = build_dataset_from_raw(
    GEN_FULL, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": v14_fingerprint(), "model": MODEL,
               "reasoning_effort": EFFORT, "composition": "prior", "version": "v15",
               "note": "scene-level roles, entity cap 8, corpus-weighted MECHANISM"},
    vocab_source=vocab.fingerprint(), seed=0, notes="v15, 6000 specs")
print(v15_id, stats["reject_reasons"])


In [ ]:

import ddi.divergence as dv
import ddi.gates as gates
from ddi.resolve import generation_records
importlib.reload(ddi.divergence); importlib.reload(ddi.gates); importlib.reload(ddi.resolve)

v15, _ = load_dataset(v15_id)
print(dv.compare(train_f, v15, match_size=True).to_string(index=False))
gates.report(v15, records=generation_records(GEN_FULL), strict=False)


In [ ]:


# ===========================================================================
# Cell 5: train, 3 seeds
# ===========================================================================
import pandas as pd
from ddi.train import train_and_eval
from ddi.experiment import log_run

BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers"}

rows = []
for seed in [0, 1, 2]:
    cfg = {**BASE, "seed": seed, "dataset": "v15", "synth_id": v15_id}
    m, p = train_and_eval(cfg, v15, dev, return_preds=True)
    log_run(cfg, m, notes="v15")
    if seed == 0:
        preds_v15 = p
    rows.append({"seed": seed, "f1": m["micro_f1_pos"], "p": m["micro_p_pos"],
                 "r": m["micro_r_pos"],
                 "f1_DrugBank": m.get("micro_f1_pos_DrugBank"),
                 "f1_MedLine": m.get("micro_f1_pos_MedLine")})
    print(f"v15 seed={seed} f1={m['micro_f1_pos']:.3f} p={m['micro_p_pos']:.3f} "
          f"r={m['micro_r_pos']:.3f}")

print(pd.DataFrame(rows)[["f1", "p", "r", "f1_DrugBank", "f1_MedLine"]]
      .agg(["mean", "std"]))
print("""
reference, same dev, 3 seeds:
  human filtered  0.790   P 0.754  R 0.829
  human full      0.800   P 0.768  R 0.835
  v14             0.379   P 0.302  R 0.511
  v14 lowrole     0.344   P 0.288  R 0.428
  v13             0.280   P 0.182  R 0.610

precision is the target. v14 seed sd 0.007, so P > 0.34 is real movement.
""")

from sklearn.metrics import classification_report
print(classification_report([r["label"] for r in dev], preds_v15,
                            labels=["MECHANISM", "EFFECT", "ADVISE", "INT"],
                            zero_division=0))
# v14 per-class precision: MECHANISM 0.23, EFFECT 0.42, ADVISE 0.30, INT 0.22